In [2]:
import os, requests, pickle
import numpy as np

np.random.seed(1337)  # For reproducibility
BASE_URL = os.getenv("BASE_URL", "http://154.57.164.64:30285")

def get_challenge(phase):
    """Fetch challenge data for specified phase"""
    r = requests.get(f"{BASE_URL}/challenge/{phase}")
    return r.json()

def submit_solutions(phase, solutions):
    """Submit solutions for validation"""
    r = requests.post(
        f"{BASE_URL}/submit/{phase}",
        json={"solutions": solutions}
    )
    return r.json()



Keys in bundle: dict_keys(['classifier', 'vectorizer', 'feature_names', 'classes'])
Model type: <class 'sklearn.naive_bayes.MultinomialNB'>
Number of features: 89972


/home/noob/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/noob/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:


def load_model():
    """Download and load the model bundle"""
    # Download model
    r = requests.get(f"{BASE_URL}/model/download")
    with open("model.pkl", "wb") as f:
        f.write(r.content)

    # Load bundle
    with open("model.pkl", "rb") as f:
        bundle = pickle.load(f)

    return bundle
# Extract good words from the downloaded model and rank by goodness score

def load_model():
    """Download and load the model bundle"""
    r = requests.get(f"{BASE_URL}/model/download")
    with open("model.pkl", "wb") as f:
        f.write(r.content)
    
    with open("model.pkl", "rb") as f:
        bundle = pickle.load(f)
    
    return bundle

# Load model
print("[*] Loading model bundle...")
bundle = load_model()
vectorizer = bundle['vectorizer']
classifier = bundle['classifier']
feature_names = bundle['feature_names']

print(f"[+] Model loaded: {len(feature_names)} features")

# Get goodness scores
log_prob_ham = classifier.feature_log_prob_[0]
log_prob_spam = classifier.feature_log_prob_[1]
goodness_scores = log_prob_ham - log_prob_spam

word_scores = list(zip(feature_names, goodness_scores))
word_scores.sort(key=lambda x: x[1], reverse=True)

# Get bottom 50 (most positive/ham-like for flipping negative→positive)
top_50_positive = word_scores[-50:][::-1]

print("\n[+] Top 50 positive words:")
for rank, (word, score) in enumerate(top_50_positive, 1):
    print(f"{rank:<6} {word:<20} {score:>15.4f}")

best_positive_words = [word for word, score in top_50_positive]

# Fetch challenge
challenge = get_challenge("blackbox")
print(f"\n[+] Reviews to flip: {len(challenge['reviews'])}")

# Test on first review
review = challenge['reviews'][0]
test_text = review['text']

print(f"\n[*] Testing review ID: {review['id']}")
print(f"    Text: {test_text[:100]}...")

# Baseline
vec_base = vectorizer.transform([test_text])
prob_base = classifier.predict_proba(vec_base)[0][1]
label_base = classifier.predict(vec_base)[0]

print(f"    Baseline: {label_base} (prob={prob_base:.4f})")

# Flip ALL 10 reviews
print("\n" + "="*60)
print("FLIPPING ALL 10 REVIEWS")
print("="*60)

solutions = []

for idx, review in enumerate(challenge['reviews']):
    review_id = review['id']
    text = review['text']
    
    print(f"\n[{idx+1}/10] Review {review_id}")
    print(f"    Original: {text[:80]}...")
    
    # Baseline
    vec_base = vectorizer.transform([text])
    prob_base = classifier.predict_proba(vec_base)[0][1]
    label_base = classifier.predict(vec_base)[0]
    
    print(f"    Baseline: {label_base} ({prob_base:.2%})")
    
    # Add words greedily until flip
    augmented = text
    words_added = 0
    max_words = challenge['max_added_words']
    
    for word in best_positive_words:
        if words_added >= max_words:
            print(f"    [!] Hit word limit ({max_words})")
            break
        
        augmented = augmented + " " + word
        vec = vectorizer.transform([augmented])
        prob = classifier.predict_proba(vec)[0][1]
        label = classifier.predict(vec)[0]
        words_added += 1
        
        if label == 'positive':
            print(f"    [✓] Flipped with {words_added} words → {label} ({prob:.2%})")
            break
    
    # Store solution
    solutions.append({
        "id": review_id,
        "augmented_text": augmented
    })

# Submit
print("\n" + "="*60)
print("SUBMITTING SOLUTIONS")
print("="*60)

result = submit_solutions("blackbox", solutions)
print(result)

[*] Loading model bundle...
[+] Model loaded: 89972 features

[+] Top 50 positive words:
1      edie                         -4.6866
2      antwone                      -4.4748
3      din                          -4.4050
4      br 10 10                     -4.2346
5      gunga                        -4.1908
6      goldsworthy                  -4.1758
7      gunga din                    -4.1605
8      yokai                        -4.0970
9      gypo                         -4.0970
10     paulie                       -4.0721
11     red sox                      -4.0115
12     visconti                     -3.9374
13     flavia                       -3.9374
14     kells                        -3.8780
15     gino                         -3.8780
16     blandings                    -3.8780
17     antwone fisher               -3.8780
18     brashear                     -3.8573
19     deathtrap                    -3.8148
20     rob roy                      -3.7703
21     lady shanghai           